# Named Entity Recognition (NER)

In [184]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np
import pandas as pd
import tensorflow as tf
tf.keras.utils.set_random_seed(33)

## 1. Dataset

In [185]:
# display original kaggle data
data = pd.read_csv("data/ner_dataset.csv", encoding = "ISO-8859-1") 
train_sents = open('data/small/train/sentences.txt', 'r').readline()
train_labels = open('data/small/train/labels.txt', 'r').readline()
print('SENTENCE:', train_sents)
print('SENTENCE LABEL:', train_labels)
print('ORIGINAL DATA:\n', data.head())
del(data, train_sents, train_labels)

SENTENCE: Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .

SENTENCE LABEL: O O O O O O B-geo O O O O O B-geo O O O O O B-gpe O O O O O

ORIGINAL DATA:
     Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O


Mỗi sample là một câu có $n$ words, thì mỗi label cũng có $n$ tags. Các tags ở label là:
* `geo`: geographical entity
* `org`: organization
* `per`: person 
* `gpe`: geopolitical entity
* `tim`: time indicator
* `art`: artifact
* `eve`: event
* `nat`: natural phenomenon
* `O`: filler word

Mỗi tag có suffix B hoặc I:
* `B`: Token begins an entity.
* `I`: Token is inside an entity.

VD có câu: __"Sharon flew to Miami on Friday"__. Các tags sẽ là:
```
Sharon B-per
flew   O
to     O
Miami  B-geo
on     O
Friday B-tim
```

Có ba tokens bắt đầu bằng B-, vì ko có multi-token entities trong sequence. Nhưng nếu thêm Sharon's last name vào: __"Sharon Floyd flew to Miami on Friday"__, các tags sẽ là
```
Sharon B-per
Floyd  I-per
flew   O
to     O
Miami  B-geo
on     O
Friday B-tim
```

"Sharon" có tag `B-per` và "Floyd" có tag `I-per`, vì đây là một inner token trong một multi-token sequence.

In [186]:
def load_data(file_path):
    with open(file_path,'r') as file:
        data = np.array([line.strip() for line in file.readlines()])
    return data

train_sentences = load_data('data/large/train/sentences.txt')
# VD câu 1 có 24 words
# ['Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .'
#  ...
# ] len=33570
train_labels = load_data('data/large/train/labels.txt')
# Label của câu 1 có 24 tags
# ['O O O O O O B-geo O O O O O B-geo O O O O O B-gpe O O O O O'
#  ...
# ] len=33570

val_sentences = load_data('data/large/val/sentences.txt') # len=7194
val_labels = load_data('data/large/val/labels.txt')

test_sentences = load_data('data/large/test/sentences.txt') # len=7194
test_labels = load_data('data/large/test/labels.txt')

## 2. Encoding
### 2.1. Encoding the sentences

In [187]:
def get_sentence_vectorizer(sentences):
    # Define TextVectorization object with the appropriate standardize parameter
    sentence_vectorizer = tf.keras.layers.TextVectorization(standardize=None)
    # Adapt the sentence vectorization object to the given sentences
    sentence_vectorizer.adapt(sentences)
    # Get the vocabulary
    vocab = sentence_vectorizer.get_vocabulary()
    
    return sentence_vectorizer, vocab

sentence_vectorizer, vocab = get_sentence_vectorizer(train_sentences)
# sentence_vectorizer sẽ chuyển một câu thành một vector có len=104 (là số words trong câu dài nhất trong train_sentences), và pad thêm "0" vào sau label của câu ngắn, chứa các ids là vị trí của words trong vocab
# vocab là một list có len=29847 là các từ có trong train_sentences
# VD câu 1 trong train_sentences sẽ đc chuyển thành vector có len=24: [1046,    6, ..., 3, 0, 0, ..., 0]
# trong đó 1046 là vị trí của word "Thousands" trong list vocab

In [188]:
sentence_vectorizer('Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .')

<tf.Tensor: shape=(24,), dtype=int64, numpy=
array([1046,    6, 1121,   18, 1832,  232,  543,    7,  528,    2,  158,
          5,   60,    9,  648,    2,  922,    6,  192,   87,   22,   16,
         54,    3])>

### 2.2. Encoding the labels

In [189]:
def get_tags(labels):
    tag_set = set() # Define an empty set
    for el in labels:
        for tag in el.split(" "):
            tag_set.add(tag)
    tag_list = list(tag_set) 
    tag_list.sort()
    return tag_list


def make_tag_map(tags):
    tag_map = {}
    for i,tag in enumerate(tags):
        tag_map[tag] = i 
    return tag_map


# Tạo một set các tags có trong labels
tags = get_tags(train_labels)
# ['B-art', 'B-eve', 'B-geo', 'B-gpe', 'B-nat', 'B-org', 'B-per', 'B-tim', 'I-art', 'I-eve', 'I-geo', 'I-gpe', 'I-nat', 'I-org', 'I-per', 'I-tim', 'O'], len=17

# Tạo một dict để map mỗi tag tới một int từ 0 tới 16
tag_map = make_tag_map(tags)
# {'B-art': 0, 'B-eve': 1, 'B-geo': 2, 'B-gpe': 3, 'B-nat': 4, 'B-org': 5, 'B-per': 6, 'B-tim': 7, 'I-art': 8, 'I-eve': 9, 'I-geo': 10, 'I-gpe': 11, 'I-nat': 12, 'I-org': 13, 'I-per': 14, 'I-tim': 15, 'O': 16}

In [190]:
def label_vectorizer(labels, tag_map):
    label_ids = [] # It can't be a numpy array yet, since each sentence has a different size

    # Each element in labels is a string of tags so for each of them:
    for element in labels:
        # Split it into single tokens. You may use .split function for strings. Be aware to split it by a blank space!
        tokens = element.split(' ')

        # Use the dictionaty tag_map passed as an argument to the label_vectorizer function
        # to make the correspondence between tags and numbers. 
        element_ids = []

        for token in tokens:
            element_ids.append(tag_map[token])

        # Append the found ids to corresponding to the current element to label_ids list
        label_ids.append(element_ids)
        
    # Pad the elements
    label_ids = tf.keras.utils.pad_sequences(label_ids, padding='post', value=-1)

    return label_ids

# Hàm label_vectorizer sẽ chuyển một label thành một vector dài 104 (là số words trong câu dài nhất trong train_sentences), và pad thêm "-1" vào sau label của câu ngắn
# VD label 'O O O O O O B-geo O O O O O B-geo O O O O O B-gpe O O O O O' sẽ đc chuyển thành [16 16 16 16 16 16  2 16 16 16 16 16  2 16 16 16 16 16  3 16 16 16 16 16 -1 -1 ... -1]
labels = label_vectorizer(train_labels, tag_map)

## 3. Build the Dataset

In [191]:
def generate_dataset(sentences, labels, sentence_vectorizer, tag_map):
    sentences_ids = sentence_vectorizer(sentences)
    labels_ids = label_vectorizer(labels, tag_map = tag_map)
    dataset = tf.data.Dataset.from_tensor_slices((sentences_ids, labels_ids))
    return dataset

train_dataset = generate_dataset(train_sentences,train_labels, sentence_vectorizer, tag_map)
# train_dataset gồm:
# 1. sentences_ids: là các vectors có len=104, chứa các ids của các words trong câu
# 2. labels_ids: là các vectors có len=104, chứa các ids của các labels trong câu

val_dataset = generate_dataset(val_sentences,val_labels,  sentence_vectorizer, tag_map)
test_dataset = generate_dataset(test_sentences, test_labels,  sentence_vectorizer, tag_map)

## 4. Model

<img src="images/ner2.png" height=300/>

In [192]:
def NER(len_tags, vocab_size, maxLen, embedding_dim = 50):
    model = tf.keras.Sequential(name = 'sequential') 

    model.add(tf.keras.layers.Input((maxLen,)))
    # Add the tf.keras.layers.Embedding layer. Do not forget to mask out the zeros!
    model.add(tf.keras.layers.Embedding(vocab_size+1, embedding_dim, mask_zero=True))

    # Add the LSTM layer. Make sure you are passing the right dimension (defined in the docstring above) 
    # and returning every output for the tf.keras.layers.LSTM layer and not the very last one.
    model.add(tf.keras.layers.LSTM(embedding_dim, return_sequences=True))

    # Add the final tf.keras.layers.Dense with the appropriate activation function. Remember you must pass the activation function itself ant not its call!
    # You must use tf.nn.log_softmax instead of tf.nn.log_softmax().
    model.add(tf.keras.layers.Dense(len_tags, activation=tf.nn.log_softmax))

    return model


def masked_loss(y_true, y_pred):
    # Calculate the loss for each item in the batch. Remember to pass the right arguments, as discussed above!
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, ignore_class=-1)

    # Use the previous defined function to compute the loss
    loss = loss_fn(y_true, y_pred)

    return  loss 


# GRADED FUNCTION: masked_accuracy
def masked_accuracy(y_true, y_pred):
    # Calculate the loss for each item in the batch.
    # You must always cast the tensors to the same type in order to use them in training. Since you will make divisions, it is safe to use tf.float32 data type.
    y_true = tf.cast(y_true, tf.float32) 
    # Create the mask, i.e., the values that will be ignored
    mask = tf.not_equal(y_true, -1)
    mask = tf.cast(mask, tf.float32) 
    # Perform argmax to get the predicted values
    y_pred_class = tf.math.argmax(y_pred, axis=-1)
    y_pred_class = tf.cast(y_pred_class, tf.float32) 
    # Compare the true values with the predicted ones
    matches_true_pred  = tf.equal(y_true, y_pred_class)
    matches_true_pred = tf.cast(matches_true_pred , tf.float32) 
    # Multiply the acc tensor with the masks
    matches_true_pred *= mask
    # Compute masked accuracy (quotient between the total matches and the total valid values, i.e., the amount of non-masked values)
    masked_acc = tf.reduce_sum(matches_true_pred)/tf.reduce_sum(mask)

    return masked_acc

In [193]:
maxLen = max([len(sentence.split(' ')) for sentence in train_sentences]) # 104
model = NER(len(tag_map), len(vocab), maxLen)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_22 (Embedding)        │ (None, 104, 50)        │     1,492,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_23 (LSTM)                  │ (None, 104, 50)        │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 104, 17)        │           867 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,513,467 (5.77 MB)

 Trainable params: 1,513,467 (5.77 MB)

 Non-trainable params: 0 (0.00 B)

In [194]:
model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss = masked_loss, metrics = [masked_accuracy])
tf.keras.utils.set_random_seed(33) ## Setting again a random seed to ensure reproducibility
BATCH_SIZE = 64
model.fit(train_dataset.batch(BATCH_SIZE),
          validation_data = val_dataset.batch(BATCH_SIZE),
          shuffle=True,
          epochs = 2)

Epoch 1/2
525/525 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - loss: 0.4594 - masked_accuracy: 0.8952 - val_loss: 0.1393 - val_masked_accuracy: 0.9573
Epoch 2/2
525/525 ━━━━━━━━━━━━━━━━━━━━ 26s 50ms/step - loss: 0.1299 - masked_accuracy: 0.9612 - val_loss: 0.1359 - val_masked_accuracy: 0.9584


In [195]:
# Convert the sentences into ids
test_sentences_id = sentence_vectorizer(test_sentences)
# Convert the labels into token ids
test_labels_id = label_vectorizer(test_labels,tag_map)
# Rename to prettify next function call
y_true = test_labels_id 
y_pred = model.predict(test_sentences_id)
print(f"The model's accuracy in test set is: {masked_accuracy(y_true,y_pred).numpy():.4f}")

225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
The model's accuracy in test set is: 0.9576


In [196]:
def predict(sentence, model, sentence_vectorizer, tag_map):
    # Convert the sentence into ids
    sentence_vectorized = sentence_vectorizer(sentence)
    # Expand its dimension to make it appropriate to pass to the model
    sentence_vectorized = tf.expand_dims(sentence_vectorized, axis=0)
    # Get the model output
    output = model(sentence_vectorized)
    # Get the predicted labels for each token, using argmax function and specifying the correct axis to perform the argmax
    outputs = np.argmax(output, axis = -1)
    # Next line is just to adjust outputs dimension. Since this function expects only one input to get a prediction, outputs will be something like [[1,2,3]]
    # so to avoid heavy notation below, let's transform it into [1,2,3]
    outputs = outputs[0] 
    # Get a list of all keys, remember that the tag_map was built in a way that each label id matches its index in a list
    labels = list(tag_map.keys()) 
    pred = [] 
    # Iterating over every predicted token in outputs list
    for tag_idx in outputs:
        pred_label = labels[int(tag_idx)]
        pred.append(pred_label)
    
    return pred

In [197]:
sentence = "Peter Parker , the White House director of trade and manufacturing policy of U.S , said in an interview on Sunday morning that the White House was working to prepare for the possibility of a second wave of the coronavirus in the fall , though he said it wouldn ’t necessarily come"
predictions = predict(sentence, model, sentence_vectorizer, tag_map)
for x,y in zip(sentence.split(' '), predictions):
    if y != 'O':
        print(x,y)

Peter B-per
Parker I-per
White B-org
House I-org
U.S B-org
Sunday B-tim
morning I-tim
White B-org
House I-org
